In [1]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
import sys
_beh_ephys_root = os.path.normpath(os.path.join(os.getcwd(), 'beh_ephys_analysis'))
if _beh_ephys_root not in sys.path:
    sys.path.insert(0, _beh_ephys_root)
from utils.beh_functions import parseSessionID
import json
from utils.capsule_migration import CAPSULE_ROOT

In [2]:
computation_params_file = CAPSULE_ROOT + '/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

In [3]:
session_data_files = [CAPSULE_ROOT + '/code/data_management/session_assets.csv']
session_assets = pd.concat([pd.read_csv(file) for file in session_data_files], ignore_index=True)
# remove nan sessions
session_assets = session_assets.dropna(subset=['session_id'])

In [4]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))
# %%
computation_id = os.getenv("CO_COMPUTATION_ID")
computation = client.computations.get_computation(computation_id=computation_id)

# For combined features + combined_unit_tbl file

In [5]:
code_feature = 'code/beh_ephys_analysis/session_combine/figure_preparation/combining_features.py'
code_tbl = 'code/beh_ephys_analysis/session_combine/figure_preparation/make_combined_unit_tbl.py'
capsule_version = '5.0' # please udpate to lastest version before use

combined_file = 'combined_unit_tbl.pkl'
feature_file = 'features_combined_beh_all.pkl'

In [6]:
# prepare data asset names
session_data_assets_names = ['raw_data', 'sorted_curated']
# prepare data asset ids, all ids in the session_assets table for these columns
session_assets = session_assets.dropna(how="all")
data_assets_ids = [
    session_assets[col].dropna().values.flatten().tolist()
    if col in session_assets.columns else []
    for col in session_data_assets_names
]

# convert list of lists to a single list
data_assets_ids = [item for sublist in data_assets_ids for item in sublist]
# drop those cannot be data asset ids, e.g. those with length less than 5
data_assets_ids = [id for id in data_assets_ids if len(id) > 5]
# remove empty strings and duplicates
# data_assets_ids = list(set([id for id in data_assets_ids if id != '']))

# append scratch data assets

# session_data_assets_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name for id in data_assets_ids]
# change to a loops that print error data
session_data_assets_names_full = []
for id in data_assets_ids:
    try:
        data_asset = client.data_assets.get_data_asset(data_asset_id=id)
        session_data_assets_names_full.append(data_asset.name)
    except Exception as e:
        print(f"Error retrieving data asset with ID {id}: {e}")
data_assets = [DataAsset(url=name) for name in session_data_assets_names_full]
# 
dp_list =[]
t = datetime(2026, 7, 9, 00, 00, 00, tzinfo=timezone.utc)
params = {}
feature_code = Code(
    url='https://github.com/AllenNeuralDynamics/aind-beh-ephys-analysis/',
    run_script=code_feature,
    version=capsule_version,
    parameters=params,
    input_data=data_assets
)

features_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name='Combine unit features for NE neurons across sessions',
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            end_date_time=t,
            output_path=feature_file,
            code=feature_code,
            notes='Combine unit features for NE neurons across sessions',
            )
dp_list.append(features_dp)

combine_code = Code(
    url='https://github.com/AllenNeuralDynamics/aind-beh-ephys-analysis/',
    run_script=code_tbl,
    version=capsule_version,
    parameters=params,
    input_data=data_assets
)

combined_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name='Combined unit table for all neurons across sessions',
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            end_date_time=t,
            output_path=combined_file,
            code=combine_code,
            notes='Combined unit table for all neurons across sessions'
            )
dp_list.append(combined_dp)

p = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)